# P92 — Optimización por enjambre de partículas

## 1. Título y paper

**Paper:** *Particle Swarm Optimization*  
**Autoría:** James Kennedy, Russell Eberhart  
**Año y venue:** 1995 · Proceedings of ICNN'95, 1942–1948  
**Nivel:** L2 · **Motor:** `pso`  
**Ficha completa:** [`P92_pso`](../../papers/foundational/P92_pso/README.md)

**Hito:** Optimiza sin gradiente con dos únicas memorias: lo mejor que ha encontrado cada individuo y lo mejor que ha encontrado el grupo.

- [doi:10.1109/ICNN.1995.488968](https://doi.org/10.1109/ICNN.1995.488968)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Muchas funciones objetivo no se pueden derivar —son simulaciones, cajas negras o tienen ruido— y los métodos de gradiente no se pueden aplicar. Las alternativas poblacionales existentes eran caras y difíciles de ajustar.
2. Ejecutar una implementación mínima de la propuesta: Un enjambre de partículas que se mueven por el espacio con una velocidad que combina inercia, atracción hacia su propio mejor histórico y atracción hacia el mejor del grupo. Sin cruce, sin mutación y sin selección.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P90
- Reynolds (1987), simulación de bandadas


## 4. Intuición

Treinta partículas volando por el espacio de soluciones. Cada una recuerda el mejor sitio donde ha estado, y todas conocen el mejor sitio que ha encontrado alguien. Su velocidad es una mezcla de inercia y atracción hacia esos dos puntos. Eso es todo el algoritmo.


## 5. Concepto mínimo

```text
v ← w·v + c₁·r₁·(mejor_personal − x) + c₂·r₂·(mejor_global − x)
x ← x + v

    w  inercia            c₁ memoria propia         c₂ memoria del grupo

Sin gradiente, sin cruce, sin selección. Solo posiciones y comparaciones.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('pso', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Bate el enjambre a la búsqueda aleatoria con el mismo presupuesto?
2. ¿Qué pasa si se quita el término social?
3. ¿Y si se quita el cognitivo?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('pso', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('pso', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El enjambre completo llega a **0,0** —el óptimo global de Rastrigin— frente al **0,6546** de la búsqueda aleatoria con las mismas 1 800 evaluaciones. Quitando el término social empeora a **2,56**: sin compartir el hallazgo son treinta búsquedas locales independientes. Quitando el cognitivo el resultado **no empeora**, y lo que cambia es la dispersión final: 0,0594 frente a 0,0081.


## 10. Comentario pedagógico

Ese último dato es el interesante y va contra la explicación de manual. En esta función el término cognitivo no acelera nada; lo que hace es conservar diversidad. La diversidad no se paga aquí porque Rastrigin tiene el óptimo en el centro, pero es el seguro contra quedarse atrapado cuando no lo está.


## 11. Error o anti-patrón deliberado

Anti-patrón: presentar los parámetros w, c₁ y c₂ como si tuvieran valores canónicos.


In [ ]:
print('Con w alto el enjambre no converge; con w bajo colapsa en la primera solucion decente.')
print('No hay teoria cerrada que los determine: se ajustan por experimento.')
print('Un articulo que reporta PSO sin declarar sus parametros no es reproducible.')

## 12. Corrección

Lo que sí se puede afirmar, con las tres configuraciones medidas:


In [ ]:
r = run_paper_lab('pso', seed=7)['result']
for nombre in ('enjambre_completo', 'solo_memoria_propia_c2_0', 'solo_memoria_del_grupo_c1_0'):
    e = r[nombre]
    print(f"{nombre:<28} mejor={e['mejor_valor']:<10} dispersion={e['dispersion_final_del_enjambre']}")
print('azar mismo presupuesto:', r['busqueda_aleatoria_mismo_presupuesto'])

## 13. Desafío guiado

Compara la dispersión final de las tres configuraciones y explica qué papel juega cada término.


In [ ]:
r = run_paper_lab('pso', seed=3)['result']
show(r)

## 14. Desafío autónomo

Aplica PSO a una función objetivo tuya que no se pueda derivar —una simulación, por ejemplo—. Barre el peso de inercia y documenta la frontera entre no converger y colapsar.


## 15. Evidencia de aprendizaje

Guarda la comparación de las tres configuraciones con su dispersión y tu explicación de para qué sirve cada memoria.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P92_pso/README.md) · evaluación formal: [`assessments/papers/P92_pso.md`](../../assessments/papers/P92_pso.md)


## 16. Cierre

El enjambre comparte a través de una variable global. La siguiente familia comparte de otra manera: dejando marcas en el propio entorno.


## 17. Conexión con el siguiente hito

- P93

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
